## Emitter-Density Simulation — Image Generation

Generates 512×512 Bayer-camera images at 50 density levels (0.01–0.50 emitters/µm²)
for three dyes (ATTO 488, ATTO 565, ATTO 647N) and three photon levels (1000, 4000, 10 000).

Each (dye, n_photons, density) condition is saved as:
- A `(n_frames, 512, 512)` uint16 TIFF stack — independent noise realisations at that density
- A companion CSV with per-frame ground-truth emitter positions (x_px, y_px) and drawn photon counts

Save root: `/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/20260630_S3MEmitterDensity/`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
import json
import time
import tifffile
from pathlib import Path

sys.path.append('../../..')
from src import IOFunctions, SpectralFunctions, MaskFunctions, PSFFunctions

IO  = IOFunctions.IO_Functions()
S_F = SpectralFunctions.Spectral_Funcs()
M_F = MaskFunctions.Mask_Functions()
PSF = PSFFunctions.PSF_Functions()

In [ ]:
# ── Camera calibration ─────────────────────────────────────────────────────────
data_folder = '../../../Camera_Calibrations/Ximea_Camera/'
gain      = IO.read_tiff(os.path.join(data_folder, 'gain.tif'))
offset    = IO.read_tiff(os.path.join(data_folder, 'offset.tif'))
variance  = IO.read_tiff(os.path.join(data_folder, 'variance.tif'))
readnoise = float(np.median(IO.read_tiff(os.path.join(data_folder, 'readnoise.tif'))))
rqe       = IO.read_tiff(os.path.join(data_folder, 'rqe.tif'))

gain_val     = float(np.median(gain))
offset_val   = float(np.median(offset))
variance_val = float(np.median(variance))
print(f'gain={gain_val:.4f}  offset={offset_val:.2f}  variance={variance_val:.4f}  readnoise={readnoise:.2f}')

In [ ]:
# ── Spectral setup ─────────────────────────────────────────────────────────────
R_base, G_base, B_base, wavelength = S_F.getpixelefficiency()
pixel_QYs = np.vstack([B_base, G_base, R_base])  # B=0, G=1, R=2
pixel_order = ['B', 'G', 'R']
filters = []

dyes   = ['ATTO 488', 'ATTO 565', 'ATTO 647N']
NA     = 1.49
pixel_size = 69.0   # nm/pixel

In [ ]:
# ── Simulation parameters ──────────────────────────────────────────────────────
image_size     = 512          # pixels (square)
n_frames       = 1000         # frames (realisations) per TIFF stack
bg_per_pixel   = 10.0          # background photoelectrons per pixel per frame

n_density_steps = 50
density_min     = 0.01        # emitters / µm²
density_max     = 0.50        # emitters / µm²
density_space   = np.linspace(density_min, density_max, n_density_steps)

n_photon_levels = [1000, 4000, 10_000]

# Image area in µm²
pixel_size_um   = pixel_size * 1e-3   # µm
image_area_um2  = (image_size * pixel_size_um) ** 2

print(f'Image: {image_size}×{image_size} px  ({image_size * pixel_size_um:.2f} µm × {image_size * pixel_size_um:.2f} µm)')
print(f'Image area: {image_area_um2:.1f} µm²')
print(f'Density range: {density_min}–{density_max} / µm²  ({n_density_steps} steps)')
print(f'N emitters at min/max density: {int(round(density_min * image_area_um2))} – {int(round(density_max * image_area_um2))}')
print(f'n_frames per stack: {n_frames}')
print(f'Total stacks: {len(density_space) * len(n_photon_levels) * len(dyes)} '
      f'({len(density_space)} densities × {len(n_photon_levels)} photon levels × {len(dyes)} dyes)')
total_gb = (
    n_density_steps * len(n_photon_levels) * len(dyes)
    * n_frames * image_size * image_size * 2 / 1e9
)
print(f'Estimated image data: {total_gb:.1f} GB')

In [ ]:
# ── 512×512 Bayer masks ────────────────────────────────────────────────────────
masks_512 = M_F.get_masks(size_x=image_size, size_y=image_size)
print('Mask channels:', list(masks_512.keys()))
for ch, m in masks_512.items():
    print(f'  {ch}: {m.sum()} True pixels  ({100 * m.mean():.1f}%)')

In [ ]:
# ── Save folder ────────────────────────────────────────────────────────────────
save_root = Path('/scratch/sycamore-asap/ASAP_Members_Other_Imaging_Data/JSB/Simulation/20260630_S3MEmitterDensity')
save_root.mkdir(parents=True, exist_ok=True)
print(f'Saving to: {save_root}')

In [ ]:
# ── Fast multi-emitter Bayer image generator ───────────────────────────────────
#
# Pipeline (same physics as gen_camera_image_stack, multi-emitter):
#   1. Accumulate Gaussian PSFs in a (W, H) buffer using local crops (fast)
#   2. Poisson noise on signal
#   3. Bayer channel filtering: Binomial(photons_at_pixel, dpe[ch]) at each pixel type
#   4. Add Poisson background PE
#   5. Camera readout: Normal(gain * PE + offset, sqrt(variance))

def _accumulate_psfs(x0_px, y0_px, n_photons, sigma_px, W, H, crop_radius=None):
    """Accumulate Gaussian PSFs using local crops. Sub-pixel accurate, ~1000× faster
    than computing full W×H Gaussians when PSF is small relative to the image."""
    if crop_radius is None:
        crop_radius = max(6, int(np.ceil(4.5 * sigma_px)))

    image = np.zeros((W, H), dtype=np.float64)

    for i in range(len(x0_px)):
        xc, yc = x0_px[i], y0_px[i]

        x1 = max(0, int(np.floor(xc)) - crop_radius)
        x2 = min(W, int(np.ceil(xc))  + crop_radius + 1)
        y1 = max(0, int(np.floor(yc)) - crop_radius)
        y2 = min(H, int(np.ceil(yc))  + crop_radius + 1)

        if x1 >= x2 or y1 >= y2:
            continue

        ix = np.arange(x1, x2, dtype=np.float64)
        iy = np.arange(y1, y2, dtype=np.float64)
        gx = np.exp(-0.5 * ((ix - xc) / sigma_px) ** 2)
        gy = np.exp(-0.5 * ((iy - yc) / sigma_px) ** 2)
        patch = np.outer(gx, gy)

        s = patch.sum()
        if s > 0:
            patch *= n_photons[i] / s   # normalise to exact photon count

        image[x1:x2, y1:y2] += patch

    return image.astype(np.float32)


def gen_multiemitter_frame(
    x0_px, y0_px, n_photons_drawn,
    dpe, sigma_px, masks, pixel_order,
    image_size, gain_val, offset_val, variance_val,
    bg_per_pixel=4.0,
):
    """
    Generate one (image_size × image_size) uint16 Bayer frame with N emitters.

    Args:
        x0_px, y0_px      : emitter positions in pixels, shape (N,)
        n_photons_drawn   : Poisson-drawn photon counts per emitter, shape (N,)
        dpe               : normalised dye-pixel efficiency, shape (n_ch,)
        sigma_px          : PSF sigma in pixels
        masks             : dict {channel_name: bool_array (W, H)}
        pixel_order       : list of channel names matching dpe order
        image_size        : W = H in pixels
        gain_val, offset_val, variance_val : scalar camera calibration values
        bg_per_pixel      : expected background photoelectrons per pixel per frame
    """
    W = H = image_size

    # 1. Expected photon image (no noise, no Bayer filtering yet)
    photon_pdf = _accumulate_psfs(x0_px, y0_px, n_photons_drawn, sigma_px, W, H)

    # 2. Poisson photon noise
    photons = np.random.poisson(photon_pdf.clip(0)).astype(np.int64)

    # 3. Bayer channel filtering: Binomial(photons, dpe[ch]) per pixel type
    photoelectrons = np.zeros((W, H), dtype=np.int64)
    for ch_idx, ch_name in enumerate(pixel_order):
        mask_ch = masks[ch_name]
        idx = np.where(mask_ch)
        photoelectrons[idx] = np.random.binomial(photons[idx], dpe[ch_idx])

    # 4. Background (uniform PE, independent of channel type)
    photoelectrons += np.random.poisson(bg_per_pixel, size=(W, H)).astype(np.int64)

    # 5. Camera readout: gain × PE + offset + N(0, variance)
    image = np.random.normal(
        gain_val * photoelectrons + offset_val,
        np.sqrt(variance_val),
    ).clip(0, 65535).astype(np.uint16)

    return image


print('Image generation functions defined.')

In [ ]:
# ── Pre-compute per-dye spectral parameters ────────────────────────────────────
dye_params = {}   # dye → {dpe, sigma_px}
for dye in dyes:
    aew, dpe_raw = S_F.get_pixel_fractions_dye_and_filters(
        [dye], filters, wavelength, pixel_QYs
    )
    dpe_arr = np.array(dpe_raw).ravel()
    dpe_norm = dpe_arr / dpe_arr.sum()
    sigma_px = PSF.sigma_PSF(float(aew), NA) / pixel_size
    dye_params[dye] = {'dpe': dpe_norm, 'sigma_px': sigma_px, 'aew_nm': float(aew)}
    print(f'{dye:12s}  aew={float(aew):.0f} nm  sigma={sigma_px:.3f} px  dpe={np.round(dpe_norm, 3)}')

In [ ]:
# ── Save metadata ──────────────────────────────────────────────────────────────
meta = {
    'image_size':       image_size,
    'pixel_size_nm':    pixel_size,
    'NA':               NA,
    'n_frames':         n_frames,
    'bg_per_pixel':     bg_per_pixel,
    'density_space':    density_space.tolist(),
    'n_photon_levels':  n_photon_levels,
    'dyes':             dyes,
    'pixel_order':      pixel_order,
    'image_area_um2':   image_area_um2,
    'gain_val':         gain_val,
    'offset_val':       offset_val,
    'variance_val':     variance_val,
    'readnoise':        readnoise,
    'dye_params':       {
        d: {
            'dpe':      dye_params[d]['dpe'].tolist(),
            'sigma_px': dye_params[d]['sigma_px'],
            'aew_nm':   dye_params[d]['aew_nm'],
        }
        for d in dyes
    },
}
(save_root / 'metadata.json').write_text(json.dumps(meta, indent=2))
print(f'Saved metadata → {save_root / "metadata.json"}')

In [ ]:
# ── Main simulation loop ───────────────────────────────────────────────────────
#
# For each (dye, n_photons, density):
#   • generate n_frames independent 512×512 Bayer images
#   • save as a (n_frames, 512, 512) uint16 TIFF stack
#   • save companion ground-truth CSV with columns:
#       frame_idx, emitter_id, x_px, y_px, n_photons_drawn

n_total_cond = len(dyes) * len(n_photon_levels) * len(density_space)
i_cond       = 0
t0_total     = time.time()

for dye in dyes:
    dye_str  = dye.replace(' ', '_')
    dp       = dye_params[dye]
    dpe      = dp['dpe']
    sigma_px = dp['sigma_px']

    for n_photon in n_photon_levels:
        ph_dir = save_root / dye_str / f'{n_photon}ph'
        ph_dir.mkdir(parents=True, exist_ok=True)

        for d_idx, density in enumerate(density_space):
            i_cond += 1

            # Number of emitters for this density
            n_emitters = max(1, int(round(density * image_area_um2)))

            stem   = f'd{d_idx:02d}_density{density:.4f}'
            tif_path = ph_dir / f'{stem}_images.tif'
            gt_path  = ph_dir / f'{stem}_gt.csv'

            # Skip if already generated (resumable)
            if tif_path.exists() and gt_path.exists():
                print(f'[{i_cond:4d}/{n_total_cond}] {dye:12s}  {n_photon:6d}ph  '
                      f'd{d_idx:02d}  N={n_emitters:4d}  (skipped)')
                continue

            t0_cond   = time.time()
            gt_rows   = []  # accumulate ground truth

            with tifffile.TiffWriter(tif_path, bigtiff=True) as tif:
                for frame_idx in range(n_frames):
                    # Random emitter positions uniformly in image (pixels)
                    x0_px = np.random.uniform(0, image_size, size=n_emitters)
                    y0_px = np.random.uniform(0, image_size, size=n_emitters)

                    # Poisson-drawn photon counts per emitter
                    n_ph_drawn = np.random.poisson(n_photon, size=n_emitters).astype(np.float32)

                    # Generate frame
                    frame = gen_multiemitter_frame(
                        x0_px, y0_px, n_ph_drawn,
                        dpe=dpe, sigma_px=sigma_px,
                        masks=masks_512, pixel_order=pixel_order,
                        image_size=image_size,
                        gain_val=gain_val, offset_val=offset_val,
                        variance_val=variance_val,
                        bg_per_pixel=bg_per_pixel,
                    )

                    tif.write(frame, contiguous=True)

                    # Ground truth for this frame
                    for em_id in range(n_emitters):
                        gt_rows.append({
                            'frame_idx':      frame_idx,
                            'emitter_id':     em_id,
                            'x_px':           x0_px[em_id],
                            'y_px':           y0_px[em_id],
                            'n_photons_drawn': int(n_ph_drawn[em_id]),
                        })

            pd.DataFrame(gt_rows).to_csv(gt_path, index=False)

            elapsed   = time.time() - t0_cond
            t_elapsed = (time.time() - t0_total) / 60
            t_remain  = (n_total_cond - i_cond) * elapsed / 60
            print(f'[{i_cond:4d}/{n_total_cond}] {dye:12s}  {n_photon:6d}ph  '
                  f'd{d_idx:02d}  N={n_emitters:4d}  '
                  f'{elapsed:.1f}s  elapsed={t_elapsed:.1f}m  ETA={t_remain:.1f}m')

print(f'\nAll done in {(time.time() - t0_total) / 60:.1f} min.')

In [ ]:
# ── Quick visual check: sample images at low, mid, high density ───────────────
check_dye      = 'ATTO 565'
check_n_photon = 4000
check_frame    = 0
check_d_idxs   = [0, 24, 49]   # first, middle, last density

fig, axs = plt.subplots(1, 3, figsize=(9, 3.5))

for ax, d_idx in zip(axs, check_d_idxs):
    density = density_space[d_idx]
    tif_path = (
        save_root
        / check_dye.replace(' ', '_')
        / f'{check_n_photon}ph'
        / f'd{d_idx:02d}_density{density:.4f}_images.tif'
    )
    if not tif_path.exists():
        ax.set_title(f'Missing: d{d_idx:02d}', fontsize=8)
        ax.axis('off')
        continue

    with tifffile.TiffFile(tif_path) as tif:
        frame = tif.pages[check_frame].asarray()

    n_em = max(1, int(round(density * image_area_um2)))
    vmin, vmax = np.percentile(frame, [1, 99])
    ax.imshow(frame, cmap='gray', vmin=vmin, vmax=vmax, origin='upper')
    ax.set_title(
        f'{density:.3f} /µm²  (N≈{n_em})', fontsize=8
    )
    ax.axis('off')

fig.suptitle(f'{check_dye}  {check_n_photon} ph  frame {check_frame}', fontsize=9)
fig.tight_layout()
plt.show()